# ChessCNN — Lichess Supervised Pretraining

Behavioural cloning on Lichess games → baseline model for AlphaZero self-play on GCP.

**Runtime:** A100 GPU — *Runtime → Change runtime type → A100 GPU*

| Step | What happens | Est. time (A100) |
|------|-------------|------------------|
| Download | ~1-3 GB Lichess PGN | 5-10 min |
| Extract | Stream games, tier-sample positions | 75-100 min |
| Train | up to 30 epochs + early stopping | 15-25 min |
| Evaluate | 60 games vs Stockfish 1320 | 12 min |

**Dataset:** 300K games → ~4.2M positions → ~126 GB RAM (of 167 GB available)  
**GPU:** A100 80 GB — batch 32,768 + gradient checkpointing uses ~20 GB VRAM  
**Speedups:** `torch.compile` (~20%) + TF32 (~10%) + LR warmup for stability

After training: download `pretrained_model.pth` and upload to GCP as `game_engine/model/best_model.pth`.

In [ ]:
import subprocess, sys

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU :', r.stdout.strip() if r.returncode == 0
      else 'NOT DETECTED — enable GPU in Runtime > Change runtime type')

print('Installing dependencies ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'chess', 'zstandard', 'tqdm'], check=True)
print('Done.')

## Configuration
Edit everything here before running the rest of the notebook.

In [ ]:
import os

# ── Data source ──────────────────────────────────────────────────────────────
LICHESS_MONTH = '2024-10'    # YYYY-MM from database.lichess.org
MAX_GAMES     = 300_000      # qualifying games  (~4.2M positions, ~126 GB RAM)
MIN_TC_SECS   = 600          # minimum base time — 600s = 10 min (thoughtful play)
SKIP_HALF     = 8            # skip first N half-moves (opening book)
SAMPLE_EVERY  = 3            # sample 1 position every N half-moves

# ── ELO tier sampling ────────────────────────────────────────────────────────
# (min_avg_elo, max_avg_elo, target_fraction_of_dataset)
TIERS = [
    (800,  1000, 0.20),   # 20% — lower club players
    (1000, 1600, 0.40),   # 40% — intermediate players
    (1600, 9999, 0.40),   # 40% — strong players
]
TIER_NAMES = ['800-1000', '1000-1600', '1600+']

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS          = 30         # hard ceiling — early stopping fires well before this
EARLY_STOP_PAT  = 5          # patience: stop after 5 consecutive epochs with no improvement
BATCH_SIZE      = 32_768     # gradient checkpointing keeps VRAM at ~20 GB on A100 80 GB
LR              = 8e-3       # linear scaling for 64× batch (512→32768); warmup stabilises
WEIGHT_DECAY    = 1e-4
VAL_SPLIT       = 0.05
NUM_WORKERS     = 12

# ── Evaluation ───────────────────────────────────────────────────────────────
STOCKFISH_ELO = 1320
N_EVAL_GAMES  = 60           # 60 games → ELO estimate ±40 ELO points

# ── Storage ──────────────────────────────────────────────────────────────────
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/chess_pretrain'
else:
    BASE_DIR = '/content/chess_pretrain'

DATA_DIR   = os.path.join(BASE_DIR, 'data')
MODEL_PATH = os.path.join(BASE_DIR, 'pretrained_model.pth')
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
print('=' * 56)
print('  ChessCNN Lichess Pretraining — Final Config')
print('=' * 56)
print(f'  Lichess month    : {LICHESS_MONTH}')
print(f'  Min time control : {MIN_TC_SECS}s  ({MIN_TC_SECS//60} min)')
print(f'  Max games        : {MAX_GAMES:,}')
print(f'  Est. positions   : ~{MAX_GAMES * 14 // 1_000_000:.1f}M  (~{MAX_GAMES * 14 * 30 // 1024**3:.0f} GB RAM)')
print(f'  Skip half-moves  : {SKIP_HALF}  (opening book)')
print(f'  Sample every     : {SAMPLE_EVERY} half-moves')
print('  ELO tiers        :')
for name, (lo, hi, frac) in zip(TIER_NAMES, TIERS):
    print(f'    {name:10s}  {int(frac*100)}%')
print('-' * 56)
print(f'  Epochs           : {EPOCHS}  (hard ceiling)')
print(f'  Early stop       : patience {EARLY_STOP_PAT} epochs')
print(f'  Batch size       : {BATCH_SIZE:,}  (grad checkpoint → ~20 GB VRAM)')
print(f'  Learning rate    : {LR}  (with 1-epoch warmup)')
print(f'  Weight decay     : {WEIGHT_DECAY}')
print(f'  Val split        : {int(VAL_SPLIT*100)}%')
print(f'  Workers          : {NUM_WORKERS}')
print(f'  Optimisations    : torch.compile + TF32 + AMP + grad checkpoint')
print('-' * 56)
print(f'  Stockfish ELO    : {STOCKFISH_ELO}')
print(f'  Eval games       : {N_EVAL_GAMES}  (±~40 ELO accuracy)')
print('=' * 56)

In [ ]:
import io, time, math, gc, types
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import (
    CosineAnnealingLR, LinearLR, SequentialLR
)
from torch.utils.checkpoint import checkpoint as grad_ckpt
import chess, chess.pgn, chess.engine
import zstandard as zstd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import urllib.request

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print(f'GPU    : {prop.name}')
    print(f'VRAM   : {prop.total_memory / 1e9:.1f} GB')

## Model
Exact copy of `game_engine/cnn.py`. Keep in sync if the architecture changes.

In [ ]:
class Mish(nn.Module):
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.squeeze    = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False), Mish(),
            nn.Linear(channels // reduction, channels, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        b, c = x.size(0), x.size(1)
        y = self.squeeze(x).view(b, c)
        return x * self.excitation(y).view(b, c, 1, 1).expand_as(x)

class ResidualBlock(nn.Module):
    def __init__(self, channels, use_se=False):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        self.act   = Mish()
        self.se    = SEBlock(channels) if use_se else None
    def forward(self, x):
        r   = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.se: out = self.se(out)
        return self.act(out + r)

class ChessCNN(nn.Module):
    def __init__(self):
        super().__init__()
        filters, blocks = 256, 20
        self.input_conv = nn.Sequential(
            nn.Conv2d(120, filters, 3, padding=1, bias=False),
            nn.BatchNorm2d(filters), Mish()
        )
        self.res_blocks = nn.ModuleList(
            [ResidualBlock(filters, use_se=(i >= 10)) for i in range(blocks)]
        )
        self.policy_head = nn.Sequential(
            nn.Conv2d(filters, 32, 1, bias=False), nn.BatchNorm2d(32), Mish(),
            nn.Flatten(), nn.Linear(32 * 8 * 8, 4672)
        )
        self.value_head = nn.Sequential(
            nn.Conv2d(filters, 1, 1, bias=False), nn.BatchNorm2d(1), Mish(),
            nn.Flatten(), nn.Linear(64, 256), Mish(), nn.Linear(256, 3)
        )
    def forward(self, x):
        x = self.input_conv(x)
        for block in self.res_blocks:
            x = block(x)
        return self.policy_head(x), self.value_head(x)

# ── TF32 — free ~10% speedup on A100 for matmul and conv ────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

model  = ChessCNN().to(device)
params = sum(p.numel() for p in model.parameters())
print(f'Parameters : {params:,}  ({params/1e6:.1f} M)')

# ── Gradient checkpointing ───────────────────────────────────────────────────
# Recomputes residual block activations during backward instead of storing them.
# Reduces activation VRAM ~4× — allows batch 32,768 within ~20 GB on A100 80 GB.
def _fwd_checkpointed(self, x):
    x = self.input_conv(x)
    for block in self.res_blocks:
        x = grad_ckpt(block, x, use_reentrant=False)
    return self.policy_head(x), self.value_head(x)

model.forward = types.MethodType(_fwd_checkpointed, model)
print('Gradient checkpointing : enabled  (~20 GB VRAM at batch 32,768)')

# ── torch.compile — ~20% throughput gain on A100 ────────────────────────────
model = torch.compile(model)
print('torch.compile          : enabled')

# Verify forward pass
with torch.no_grad():
    _p, _v = model(torch.randn(2, 120, 8, 8, device=device))
print(f'Policy out : {tuple(_p.shape)}')
print(f'Value out  : {tuple(_v.shape)}')

## Encoders
Pure-Python equivalents of the C++ engine — board → 120-plane tensor, move → 4672 index.

In [ ]:
_PIECE_TYPES = [
    chess.PAWN, chess.KNIGHT, chess.BISHOP,
    chess.ROOK, chess.QUEEN,  chess.KING
]

def _fill_frame(tensor, board, offset):
    """14 planes at offset: 6 white pieces, 6 black pieces, 2 repetition."""
    for i, pt in enumerate(_PIECE_TYPES):
        for sq in board.pieces(pt, chess.WHITE):
            tensor[offset + i,     sq // 8, sq % 8] = 1.0
        for sq in board.pieces(pt, chess.BLACK):
            tensor[offset + 6 + i, sq // 8, sq % 8] = 1.0
    if board.is_repetition(2): tensor[offset + 12, :, :] = 1.0
    if board.is_repetition(3): tensor[offset + 13, :, :] = 1.0

def board_to_tensor(board, history=None):
    """
    (120, 8, 8) float32 — matches chess_board.cpp to_tensor() exactly.
    history: list[chess.Board] most-recent-first, up to 7 boards.
    """
    t = np.zeros((120, 8, 8), dtype=np.float32)
    for i, b in enumerate([board] + list(history or [])[:7]):
        _fill_frame(t, b, i * 14)
    if board.has_kingside_castling_rights(chess.WHITE):  t[112] = 1.0
    if board.has_queenside_castling_rights(chess.WHITE): t[113] = 1.0
    if board.has_kingside_castling_rights(chess.BLACK):  t[114] = 1.0
    if board.has_queenside_castling_rights(chess.BLACK): t[115] = 1.0
    if board.ep_square is not None:
        t[116, board.ep_square // 8, board.ep_square % 8] = 1.0
    if board.turn == chess.BLACK: t[117] = 1.0
    t[118] = board.fullmove_number / 512.0
    t[119] = board.halfmove_clock  / 100.0
    return t

_KNIGHT_MAP = {d: i for i, d in enumerate(
    [(1,2),(2,1),(2,-1),(1,-2),(-1,-2),(-2,-1),(-2,1),(-1,2)]
)}

def move_to_index(move):
    """
    AlphaZero 4672 encoding: src*73 + plane.
    Matches mcts_engine.h move_to_index() exactly.
    """
    src = move.from_square
    df  = chess.square_file(move.to_square) - chess.square_file(src)
    dr  = chess.square_rank(move.to_square) - chess.square_rank(src)
    if move.promotion and move.promotion != chess.QUEEN:
        piece = {chess.KNIGHT: 0, chess.BISHOP: 1, chess.ROOK: 2}[move.promotion]
        return src * 73 + 64 + (0 if df < 0 else 1 if df == 0 else 2) * 3 + piece
    adf, adr = abs(df), abs(dr)
    if (adf, adr) in ((1,2),(2,1)):
        return src * 73 + 56 + _KNIGHT_MAP[(df, dr)]
    if   df == 0:           d = 0 if dr>0 else 4;  dist = adr-1
    elif dr == 0:           d = 2 if df>0 else 6;  dist = adf-1
    elif df>0 and dr>0:     d = 1;                 dist = adf-1
    elif df<0 and dr>0:     d = 7;                 dist = adf-1
    elif df>0 and dr<0:     d = 3;                 dist = adf-1
    else:                   d = 5;                 dist = adf-1
    return src * 73 + d * 7 + dist

# Sanity checks
_b = chess.Board()
_t = board_to_tensor(_b)
assert _t.shape == (120, 8, 8)
assert _t[0, 1, :].sum() == 8
assert move_to_index(chess.Move.from_uci('e2e4')) == chess.E2 * 73 + 0 * 7 + 1
assert move_to_index(chess.Move.from_uci('g1f3')) == chess.G1 * 73 + 56 + _KNIGHT_MAP[(-1,2)]
print('board_to_tensor  OK')
print('move_to_index    OK')

## Data Pipeline
Downloads one Lichess month, streams positions into three ELO tiers, then resamples to the configured 20/40/40 split.

In [ ]:
def download_lichess(month, dest_dir):
    url  = (f'https://database.lichess.org/standard/'
            f'lichess_db_standard_rated_{month}.pgn.zst')
    path = os.path.join(dest_dir, f'lichess_{month}.pgn.zst')
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'Already downloaded  ({size_mb:.0f} MB) : {path}')
        return path

    print(f'Downloading: {url}')
    bar = tqdm(unit='MB', unit_scale=True, desc='Download')
    last = [0]
    def hook(count, block, total):
        downloaded = count * block
        bar.total = total
        bar.update(downloaded - last[0])
        last[0] = downloaded
    urllib.request.urlretrieve(url, path, hook)
    bar.close()
    print(f'Saved ({os.path.getsize(path)/1e6:.0f} MB): {path}')
    return path

pgn_path = download_lichess(LICHESS_MONTH, DATA_DIR)

In [ ]:
def get_tier(avg_elo):
    for i, (lo, hi, _) in enumerate(TIERS):
        if lo <= avg_elo < hi:
            return i
    return -1   # below 800 or other

def extract_positions(pgn_zst_path):
    """
    Stream-decompress Lichess PGN and extract positions.
    Filters: 10-min+ time control, avg ELO in a valid tier (800+).
    Returns arrays: states (N,120,8,8), policies (N,), values (N,), tier_labels (N,).
    """
    dctx = zstd.ZstdDecompressor()

    states, policies, values, tier_labels = [], [], [], []
    games_ok   = 0
    games_skip = 0
    tier_game_counts = [0, 0, 0]

    with open(pgn_zst_path, 'rb') as fh:
        with dctx.stream_reader(fh) as reader:
            text = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')

            pbar = tqdm(total=MAX_GAMES, desc='Games extracted', unit='game')

            while True:
                if MAX_GAMES and games_ok >= MAX_GAMES:
                    break

                game = chess.pgn.read_game(text)
                if game is None:
                    break

                hdr = game.headers

                # Time-control filter (>= 10 min)
                try:
                    base = int(hdr.get('TimeControl', '0').split('+')[0])
                    if base < MIN_TC_SECS:
                        games_skip += 1
                        continue
                except ValueError:
                    games_skip += 1
                    continue

                # ELO filter and tier assignment
                try:
                    w_elo = int(hdr.get('WhiteElo', '0'))
                    b_elo = int(hdr.get('BlackElo', '0'))
                    avg   = (w_elo + b_elo) / 2
                    tier  = get_tier(avg)
                    if tier < 0:
                        games_skip += 1
                        continue
                except ValueError:
                    games_skip += 1
                    continue

                # Result filter
                result = hdr.get('Result', '*')
                if result not in ('1-0', '0-1', '1/2-1/2'):
                    games_skip += 1
                    continue

                board   = game.board()
                history = []
                half    = 0

                for move in game.mainline_moves():
                    half += 1
                    if half > SKIP_HALF and half % SAMPLE_EVERY == 0:
                        if result == '1/2-1/2':
                            vc = 1
                        elif ((result == '1-0'  and board.turn == chess.WHITE) or
                              (result == '0-1'  and board.turn == chess.BLACK)):
                            vc = 0
                        else:
                            vc = 2
                        states.append(board_to_tensor(board, history[-7:]))
                        policies.append(move_to_index(move))
                        values.append(vc)
                        tier_labels.append(tier)

                    history.insert(0, board.copy())
                    board.push(move)

                games_ok += 1
                tier_game_counts[tier] += 1
                pbar.update(1)
                pbar.set_postfix({
                    TIER_NAMES[0]: tier_game_counts[0],
                    TIER_NAMES[1]: tier_game_counts[1],
                    TIER_NAMES[2]: tier_game_counts[2],
                    'pos': len(states),
                })

            pbar.close()

    print(f'\nExtracted {len(states):,} positions from {games_ok:,} games')
    print(f'Skipped   {games_skip:,} games (ELO / time-control / result filter)')
    print(f'Games per tier : {[f"{TIER_NAMES[i]}={tier_game_counts[i]:,}" for i in range(3)]}')
    return (
        np.array(states,      dtype=np.float32),
        np.array(policies,    dtype=np.int32),
        np.array(values,      dtype=np.int8),
        np.array(tier_labels, dtype=np.int8),
    )


cache = os.path.join(
    DATA_DIR,
    f'raw_{LICHESS_MONTH}_tc{MIN_TC_SECS}_games{MAX_GAMES}.npz'
)

if os.path.exists(cache):
    print(f'Loading cache: {cache}')
    d = np.load(cache)
    raw_states = d['states']; raw_policies = d['policies']
    raw_values = d['values']; raw_tiers    = d['tiers']
    print(f'Loaded {len(raw_states):,} positions')
else:
    raw_states, raw_policies, raw_values, raw_tiers = extract_positions(pgn_path)
    np.savez_compressed(cache,
        states=raw_states, policies=raw_policies,
        values=raw_values, tiers=raw_tiers)
    print(f'Saved to cache: {cache}')

In [ ]:
# Resample raw positions to the configured 20 / 40 / 40 tier split
rng = np.random.default_rng(42)
total_raw = len(raw_states)

# Compute target count per tier (last tier absorbs rounding)
targets = [int(total_raw * frac) for _, _, frac in TIERS]
targets[-1] = total_raw - sum(targets[:-1])

print('Tier resampling')
print('-' * 52)
print(f'{"Tier":<12} {"Available":>10} {"Target":>10} {"Selected":>10} {"Actual%":>8}')
print('-' * 52)

selected_idx = []
for t_idx, (target, name) in enumerate(zip(targets, TIER_NAMES)):
    idx = np.where(raw_tiers == t_idx)[0]
    n   = min(target, len(idx))
    chosen = rng.choice(idx, size=n, replace=False)
    selected_idx.append(chosen)
    pct = n / total_raw * 100
    flag = '' if len(idx) >= target else '  (insufficient — used all available)'
    print(f'{name:<12} {len(idx):>10,} {target:>10,} {n:>10,} {pct:>7.1f}%{flag}')

print('-' * 52)

all_idx = np.concatenate(selected_idx)
rng.shuffle(all_idx)

all_states   = raw_states[all_idx]
all_policies = raw_policies[all_idx]
all_values   = raw_values[all_idx]

wdl = np.bincount(all_values.astype(np.uint8), minlength=3)
N   = len(all_states)
print(f'\nFinal dataset : {N:,} positions')
print(f'WDL           : W={wdl[0]:,} ({wdl[0]/N*100:.1f}%)  '
      f'D={wdl[1]:,} ({wdl[1]/N*100:.1f}%)  '
      f'L={wdl[2]:,} ({wdl[2]/N*100:.1f}%)')

# Free raw arrays
del raw_states, raw_policies, raw_values, raw_tiers
gc.collect()

## Training
- **Policy loss** — CrossEntropy(logits, move_index): learn which move to play
- **Value loss** — CrossEntropy(logits, wdl_class): learn game outcome (W=0, D=1, L=2)
- Mixed-precision (AMP) + gradient clipping + cosine LR decay

In [ ]:
class LichessDataset(Dataset):
    def __init__(self, states, policies, values):
        self.states   = torch.from_numpy(states)
        self.policies = torch.from_numpy(policies.astype(np.int64))
        self.values   = torch.from_numpy(values.astype(np.int64))
    def __len__(self): return len(self.states)
    def __getitem__(self, i):
        return self.states[i], self.policies[i], self.values[i]

dataset  = LichessDataset(all_states, all_policies, all_values)
n_val    = max(1, int(len(dataset) * VAL_SPLIT))
n_train  = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val])

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=True, prefetch_factor=4)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=True, prefetch_factor=4)

print(f'Train : {n_train:,} positions  /  {len(train_dl)} batches per epoch')
print(f'Val   : {n_val:,} positions  /  {len(val_dl)} batches per epoch')
print(f'Batch : {BATCH_SIZE}  |  Workers : {NUM_WORKERS}  |  Device : {device}')

In [ ]:
optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_dl)
warmup_steps = len(train_dl)            # 1 epoch linear warmup

# Linear warmup (1 epoch) → cosine decay to 1e-5
warmup_sched = LinearLR(optimizer, start_factor=0.01, end_factor=1.0,
                         total_iters=warmup_steps)
cosine_sched = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps,
                                  eta_min=1e-5)
scheduler    = SequentialLR(optimizer,
                             schedulers=[warmup_sched, cosine_sched],
                             milestones=[warmup_steps])

amp_device = 'cuda' if torch.cuda.is_available() else 'cpu'
scaler     = torch.amp.GradScaler(amp_device)

log = {'train_p': [], 'train_v': [], 'val_p': [], 'val_v': []}
best_val      = float('inf')
no_improve    = 0
stopped_epoch = EPOCHS

baseline_p = math.log(4672)
baseline_v = math.log(3)

print('=' * 80)
print(f'  Training  |  up to {EPOCHS} epochs  |  early stop patience = {EARLY_STOP_PAT}')
print(f'  {len(train_dl)} batches/epoch  |  batch {BATCH_SIZE:,}  |  '
      f'lr {LR} w/ 1-epoch warmup  |  TF32 + AMP + grad-ckpt')
print(f'  Random baseline  policy {baseline_p:.3f}  value {baseline_v:.3f}')
print('=' * 80)
print(f'{"Epoch":>6} {"Train-P":>9} {"Train-V":>9} {"Val-P":>9} {"Val-V":>9}'
      f' {"LR":>9} {"Time":>7}')
print('-' * 80)

def run_epoch(loader, train=True):
    model.train(train)
    p_sum = v_sum = n = 0
    pbar = tqdm(loader, leave=False,
                desc='  train' if train else '    val',
                unit='batch')
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for states, pol_tgt, val_tgt in pbar:
            states  = states.to(device,  non_blocking=True)
            pol_tgt = pol_tgt.to(device, non_blocking=True)
            val_tgt = val_tgt.to(device, non_blocking=True)
            if train: optimizer.zero_grad()
            with torch.amp.autocast(amp_device):
                pol_out, val_out = model(states)
                p_loss = F.cross_entropy(pol_out, pol_tgt)
                v_loss = F.cross_entropy(val_out, val_tgt)
                loss   = p_loss + v_loss
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            p_sum += p_loss.item(); v_sum += v_loss.item(); n += 1
            pbar.set_postfix(p=f'{p_sum/n:.3f}', v=f'{v_sum/n:.3f}')
    return p_sum / n, v_sum / n


for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tp, tv = run_epoch(train_dl, train=True)
    vp, vv = run_epoch(val_dl,   train=False)
    elapsed = time.time() - t0

    log['train_p'].append(tp); log['train_v'].append(tv)
    log['val_p'].append(vp);   log['val_v'].append(vv)

    val_total = vp + vv
    lr_now    = scheduler.get_last_lr()[0]

    if val_total < best_val:
        best_val   = val_total
        no_improve = 0
        torch.save({'model_state_dict': model.state_dict(),
                    'epoch': epoch,
                    'val_policy_loss': vp,
                    'val_value_loss':  vv}, MODEL_PATH)
        tag = '  ✓ saved'
    else:
        no_improve += 1
        tag = f'  no improve ({no_improve}/{EARLY_STOP_PAT})'

    print(f'{epoch:>6}/{EPOCHS}  {tp:>9.4f}  {tv:>9.4f}  {vp:>9.4f}  '
          f'{vv:>9.4f}  {lr_now:>9.2e}  {elapsed:>5.0f}s{tag}')

    if no_improve >= EARLY_STOP_PAT:
        stopped_epoch = epoch
        print(f'\nEarly stop — val loss stagnant for {EARLY_STOP_PAT} consecutive epochs.')
        break

print('-' * 80)
print(f'Stopped at epoch  : {stopped_epoch}  (ceiling {EPOCHS})')
print(f'Best val loss     : {best_val:.4f}')
print(f'Policy  {baseline_p:.3f} → {min(log["val_p"]):.4f}'
      f'  ({(1 - min(log["val_p"]) / baseline_p) * 100:.1f}% below random)')
print(f'Value   {baseline_v:.3f} → {min(log["val_v"]):.4f}'
      f'  ({(1 - min(log["val_v"]) / baseline_v) * 100:.1f}% below random)')

## Training Curves

In [ ]:
epochs_x = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, key, title, baseline in [
    (axes[0], 'p', 'Policy Loss (cross-entropy)', baseline_p),
    (axes[1], 'v', 'Value Loss — WDL (cross-entropy)', baseline_v),
]:
    ax.plot(epochs_x, log[f'train_{key}'], 'b-o', lw=2, label='Train')
    ax.plot(epochs_x, log[f'val_{key}'],   'r-o', lw=2, label='Val')
    ax.axhline(baseline, color='gray', lw=1, ls='--', label=f'Random ({baseline:.2f})')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(list(epochs_x))

plt.suptitle(
    f'Lichess {LICHESS_MONTH}  |  10-min+ games  |  '
    f'Tiers 20/40/40  |  {N:,} positions',
    fontsize=11
)
plt.tight_layout()
curve_path = os.path.join(BASE_DIR, 'training_curves.png')
plt.savefig(curve_path, dpi=150)
plt.show()
print(f'Saved: {curve_path}')

## ELO Evaluation vs Stockfish
Greedy policy (argmax over legal moves, **no MCTS**).  
This is the floor — actual ELO will be significantly higher once MCTS search is added in self-play.

In [ ]:
import subprocess as _sp
if _sp.run(['which', 'stockfish'], capture_output=True).returncode != 0:
    print('Installing Stockfish ...')
    _sp.run(['apt-get', 'install', '-y', 'stockfish'], check=True, capture_output=True)

# Load best checkpoint
ckpt = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')
print(f'Val policy loss : {ckpt["val_policy_loss"]:.4f}')
print(f'Val value  loss : {ckpt["val_value_loss"]:.4f}')


def greedy_move(board, history):
    """Pick the legal move with the highest policy logit."""
    t = torch.from_numpy(board_to_tensor(board, history)).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = model(t)
    logits = logits[0].cpu()
    return max(board.legal_moves, key=lambda m: logits[move_to_index(m)].item())


def play_vs_sf(sf_engine, model_white, sf_elo):
    board   = chess.Board()
    history = []
    opt = sf_engine.options.get('UCI_Elo')
    elo = max(sf_elo, opt.min or 0) if opt else sf_elo
    sf_engine.configure({'UCI_LimitStrength': True, 'UCI_Elo': elo})
    for _ in range(400):
        if board.is_game_over(): break
        if (board.turn == chess.WHITE) == model_white:
            move = greedy_move(board, history)
        else:
            move = sf_engine.play(board, chess.engine.Limit(time=0.1)).move
        history.insert(0, board.copy())
        board.push(move)
    res = board.result()
    if res == '1-0': return 'win'  if model_white else 'loss'
    if res == '0-1': return 'loss' if model_white else 'win'
    return 'draw'


print(f'\nPlaying {N_EVAL_GAMES} games vs Stockfish {STOCKFISH_ELO} ...')
print(f'(alternating colours — {N_EVAL_GAMES//2} as White, {N_EVAL_GAMES//2} as Black)')
print()

sf     = chess.engine.SimpleEngine.popen_uci('stockfish')
tally  = {'win': 0, 'draw': 0, 'loss': 0}
pbar   = tqdm(range(N_EVAL_GAMES), desc='Eval games', unit='game')

for i in pbar:
    r = play_vs_sf(sf, model_white=(i % 2 == 0), sf_elo=STOCKFISH_ELO)
    tally[r] += 1
    pbar.set_postfix(W=tally['win'], D=tally['draw'], L=tally['loss'])

sf.quit()

score = tally['win'] + 0.5 * tally['draw']
pct   = score / N_EVAL_GAMES
if 0 < pct < 1:
    est = STOCKFISH_ELO - 400 * math.log10(1 / pct - 1)
    elo_str = f'~{est:.0f}'
elif pct == 0:
    elo_str = f'<{STOCKFISH_ELO - 800}'
else:
    elo_str = f'>{STOCKFISH_ELO + 800}'

print()
print('=' * 52)
print(f'  Result vs Stockfish {STOCKFISH_ELO}')
print(f'  W={tally["win"]}  D={tally["draw"]}  L={tally["loss"]}')
print(f'  Score : {score:.1f} / {N_EVAL_GAMES}  ({pct:.1%})')
print(f'  ELO   : {elo_str}  (greedy, no MCTS — floor estimate)')
print('=' * 52)
print('  With MCTS search added in self-play, ELO will be substantially higher.')

## Save & Download
Download the model, then upload it to GCP as `chess_ai/game_engine/model/best_model.pth`.

In [ ]:
from google.colab import files as _cf

size_mb = os.path.getsize(MODEL_PATH) / 1e6
print(f'Model : {MODEL_PATH}  ({size_mb:.0f} MB)')
print('Downloading ...')
_cf.download(MODEL_PATH)

print()
print('Next steps:')
print('  1. Upload pretrained_model.pth to your GCP instance')
print('  2. cp pretrained_model.pth chess_ai/game_engine/model/best_model.pth')
print('  3. cd chess_ai && python game_engine/main.py')
print('     AlphaZero self-play resumes from this pretrained baseline.')